# 11.5 全域變數（global）在 APCS 競賽中的使用準則與除錯防禦

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_11-5_global_variable_and_debugging.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 11.1 函數定義與呼叫、11.2 函數回傳值、11.3 參數傳遞與副作用防禦，以及 11.4 變數作用域與遮蔽。

---

### 學習導覽：解鎖全域黑科技與競賽防禦護甲

在上一單元 11.4 中，我們學習了函數內部的「區域小王國（Local Scope）」，並見識到如果在函數內部嘗試直接改寫外部的全域變數，Python 直譯器會立刻拋出經典的 `UnboundLocalError`。

然而，在 APCS 程式設計競賽與演算法實戰中，我們常常會遇到以下棘手場景：
- **全域極值維護**：在搜尋演算法（如深度優先搜尋 DFS、回溯法）探索數百條可能路徑時，如何隨時記錄「目前找到的最小步數」或「最高得分」，而不需要在每一層函數之間傳遞繁瑣的參數？
- **巨型地圖共享**：面對 $1000 \times 1000$ 的龐大二維網格地圖，如何讓多個子函數共享同一張地圖，而不會因為頻繁傳遞或誤複製引發記憶體超限（MLE）或執行超時（TLE）？
- **世紀重測慘劇**：為什麼許多初學選手在自己電腦上測試單筆範例完全正確，一送上 APCS 線上評判系統（OJ）卻因為「跨測資全域變數未清空」直接吞下 0 分 Wrong Answer (WA)？

在本單元中，我們將以 6 個清晰嚴謹的階梯，深入探討全域變數在競賽中的強大威力與防禦除錯：
1. **11.5.1 `global` 關鍵字**：在區域內宣告並修改全域變數的值（打破作用域城牆的特許令）。
2. **11.5.2 APCS 實戰場景一**：全域最佳解累計計數器（`ans = min(ans, ...)` 的維護模型）。
3. **11.5.3 APCS 實戰場景二**：大型地圖 / 圖論鄰接矩陣的全域共享存取與唯讀優化。
4. **11.5.4 可變物件免 `global` 原理**：深入剖析為何串列/字典的原處修改不需要宣告 `global`。
5. **11.5.5 濫用 `global` 的隱患**：狀態污染與跨測資未清空的致命傷（診斷 0 分 bug）。
6. **11.5.6 競賽除錯守則**：多筆測資輸入時，如何在迴圈開頭正確落實黃金重置（Reset）SOP！

讓我們裝備好這份競技利器，邁向既快速又堅不可摧的程式架構！

### 11.5.1 `global` 關鍵字：在區域內宣告並修改全域變數的值

#### 1. 生活故事比喻：廣場中央的公共計分大看板
想像一座城市的中央廣場上，樹立著一塊巨大的電子計分看板（全域變數 `score = 0`），全城的居民隨時抬頭都能看到看板上的數字（讀取全域變數）。在廣場旁邊的裁判室裡，有一位計分裁判員（函數）。
如果裁判員只想在自己的筆記本上記分，他在室內隨意寫 `score = 10`，那只是他的私人紙張（區域變數）。但如果今天比賽得分了，他想要拿起控制器直接「改寫廣場中央那面大看板」，他就不能偷偷動手。如果他沒出示市政廳頒發的「特許工作證」，守衛就會認定他意圖不明並強制攔截（引發 `UnboundLocalError`）。
只有當裁判員在工作開始前大聲出示證件說：**「`global score`！我要動用的是廣場中央那面大看板！」**守衛才會放行，允許他在函數內部直接更新全城皆可見的大看板數值。

#### 2. 底層運作機制：打破區域繫結的宣告語法
在 Python 的編譯期語法分析中，只要直譯器在函數內部看到任何對變數名稱的賦值語句（例如 `x = 10` 或 `x += 1`），就會預設將該變數名稱標記為「區域命名空間（Local Namespace）」。
若我們要在函數內部修改「全域命名空間（Global Namespace）」中已存在的變數，就必須使用 **`global 變數名稱`** 關鍵字：
- **宣告作用**：通知 Python 直譯器「請跳過區域命名空間，直接將此名稱連結至最外層全域命名空間中的同名變數」。
- **生效範圍**：僅在當前宣告的該函數內部有效。宣告後，該函數內所有對該變數的讀取、修改與重新賦值，都會直接作用於全域變數。

#### 3. 初學者常見陷阱與語法地雷
- **陷阱一：單行賦值語法錯誤**  
  初學同學常順手寫成：`global score = 100`。這是嚴重的語法錯誤（`SyntaxError: invalid syntax`）！`global` 是宣告關鍵字，不可與賦值符號放在同一行。正確寫法必須拆為兩行：先宣告 `global score`，下一行再寫 `score = 100`。
- **陷阱二：宣告前就提前使用**  
  若在寫 `global x` 之前，函數上方已經有讀取 `x` 的行為（如 `print(x)`），Python 會拋出 `SyntaxError: name 'x' is used prior to global declaration`。`global` 宣告必須放在函數內使用該變數的最頂端！

#### 4. APCS 實戰視野
在 APCS 競賽中，當你需要維護一個全域計數器、狀態旗標或最佳解時，`global` 能讓你免去在多層函數之間傳遞參數並層層 return 的複雜結構，大幅提升考場上的編碼速度與簡潔度。

In [ ]:
# 範例 11.5.1：global 關鍵字的正確宣告與全域變數改寫

# 1. 在全域定義公共金幣箱
gold_coins = 100

def collect_coins(earned):
    # 宣告 global，明確指出要修改全域的 gold_coins
    global gold_coins
    
    print(f"  [函數內部] 收集前全域金幣數: {gold_coins}")
    gold_coins += earned  # 直接改寫全域變數
    print(f"  [函數內部] 收集後全域金幣數: {gold_coins}")

# 主程式呼叫
print(f"[主程式] 初始金幣: {gold_coins}")
collect_coins(50)
print(f"[主程式] 呼叫後金幣成功更新為: {gold_coins}")

collect_coins(30)
print(f"[主程式] 再次呼叫後金幣為: {gold_coins}")

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.1
# 任務說明：
# 某位同學正在設計一個全域計程車跳表系統。
# 全域變數 `fare` 初始為 85 元（基本起跳價）。
# 每次呼叫 `add_mileage(extra_fare)` 時，必須修改全域的 `fare`。
# 請補齊程式碼中的 `___`：
# 1. 在函數最頂端以 global 宣告要修改的變數。
# 2. 將 extra_fare 累加至全域 fare 中。
# ==========================================

fare = 85

def add_mileage(extra_fare):
    # 提示：宣告要修改全域變數 fare
    ___ fare
    
    # 提示：將額外車資累加進全域變數
    fare += ___
    print(f"  [跳表機] 當前累計車資: {fare} 元")

# 主程式測試
print(f"[主程式] 起跳車資: {fare} 元")
add_mileage(20)
add_mileage(30)
print(f"[主程式] 最終應付車資: {fare} 元")  # 預期輸出: 135 元

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.1
# 任務說明：
# 請設計一個全域點擊記錄系統：
# 1. 在全域宣告變數 `total_clicks = 0`。
# 2. 定義函數 `record_clicks(count)`：
#    - 透過 global 宣告修改全域的 total_clicks。
#    - 將 count 累加到 total_clicks 中。
# 3. 主程式依據測試資料連續呼叫兩次，並印出最終 total_clicks 的數值。
#
# 【公開測試資料 1】
# 呼叫順序：record_clicks(5)，接著 record_clicks(3)
# 預期輸出：
# 全域總點擊數: 8
#
# 【公開測試資料 2】
# 呼叫順序：record_clicks(12)，接著 record_clicks(28)
# 預期輸出：
# 全域總點擊數: 40
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.1
# 任務說明：
# 請設計一個全域太空船防護罩系統：
# 1. 全域變數 `shield_energy = 100`（防護罩初始能量 100%）。
# 2. 定義函數 `take_damage(damage)`：
#    - 使用 global 改寫 shield_energy。
#    - 扣除傷害 damage，但若能量小於 0，防護罩必須鎖定在 0（不能變成負數）。
# 3. 定義函數 `repair_shield(amount)`：
#    - 使用 global 改寫 shield_energy。
#    - 增加修復量 amount，但能量上限最高只能達到 100（不可超過 100%）。
# 4. 在主程式中模擬受到傷害與修復過程，並印出各階段全域防護罩能量。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.5.2 APCS 競賽實戰場景一：全域最佳解累計計數器（`ans = min(ans, ...)`）

#### 1. 生活故事比喻：全場最低價尋寶雷達
想像你是一位拍賣會尋寶獵人，總部派出手下進入多個展覽廳尋找「價格最低的稀世珍寶」。每當手下在某個展廳發現一件寶物時，手下不需要把整個展廳的幾百件商品全部搬回總部，也不需要在每扇門之間互相大喊溝通。
總部的白板上只寫著一個數字：**「目前全場最低價」**（初始設為無限大 $\infty$）。
手下一旦在某個房間發現一件定價 500 元的寶物，他立刻透過無線電更新總部白板：`best_price = min(best_price, 500)`。下一個手下在另一個房間發現 800 元的寶物，比對後發現沒有更低，白板維持 500；再下一個手下發現 350 元的寶物，立刻將白板更新為 350！
搜尋結束時，總部白板上的數字就是全場最便宜的唯一正解。

#### 2. 底層運作機制：極值全域維護模型
在 APCS 的搜尋題型中（例如 DFS、遞迴枚舉、狀態路徑探索），我們通常需要一個變數來記錄全局最優解：
- **求最小值（Minimization）**：
  ```python
  best_cost = float('inf')  # 初始設定為無限大
  def search_path(current_cost):
      global best_cost
      best_cost = min(best_cost, current_cost)
  ```
- **求最大值（Maximization）**：
  ```python
  best_profit = -float('inf')  # 或初始設為 0（若收益非負）
  def search_path(current_profit):
      global best_profit
      best_profit = max(best_profit, current_profit)
  ```
這樣設計的優勢在於：子函數只要專注於評估當前路徑，一旦到達終點或葉節點，直接以一行 `min/max` 更新全域最佳解，完全不必把最佳解層層 `return` 往上交接，大幅降低思考負擔。

#### 3. 初學者常見陷阱：極值初始化數值設錯
初學同學最常犯的致命邏輯錯誤是：**找最小值時，把全域變數初始化為 0**！
如果 `best_cost = 0`，當後續探索找到成本為 15、20、8 的路徑時，`min(0, 15)`、`min(0, 8)` 的計算結果永遠是 0！你的程式最後就會自信滿滿地輸出 0，引發 Wrong Answer！
請牢記鐵律：
- 找最小值：初始值必須設為**極大值**（如 `float('inf')` 或題目上限 $10^9$）。
- 找最大值：初始值必須設為**極小值**（如 `-float('inf')` 或 `-1`）。

#### 4. APCS 實戰視野
在 APCS 歷屆實作三級題（如最短路徑步數、最小切片成本、背包最大價值）中，全域最佳解累計計數器是撰寫搜尋演算法時最簡潔俐落的黃金架構，能讓程式碼行數大幅縮減。

In [ ]:
# 範例 11.5.2：維護全域最佳解計數器（尋找最低航程耗時）

# 初始設定全域最佳耗時為無限大
min_flight_time = float('inf')
best_airline = ""

def evaluate_flight(airline_name, flight_hours):
    global min_flight_time, best_airline
    
    print(f"  [航程評估] 正在審查 {airline_name}，耗時: {flight_hours} 小時")
    # 若當前航程耗時比歷史最低耗時更短，則更新全域最佳解
    if flight_hours < min_flight_time:
        min_flight_time = flight_hours
        best_airline = airline_name
        print(f"    ⭐ 刷新全域最佳紀錄！最低耗時更新為: {min_flight_time} 小時 ({best_airline})")

# 模擬評估多條航線
evaluate_flight("星空航空", 14.5)
evaluate_flight("陽光航空", 18.0)
evaluate_flight("極速航空", 12.2)
evaluate_flight("藍天航空", 13.0)

print(f"\n[最終結果] 最推薦航線: {best_airline}，最低耗時: {min_flight_time} 小時")

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.2
# 任務說明：
# 某位同學要挑選全校運動會中跳高成績最高的選手。
# 全域變數 `highest_jump` 初始設為 0.0。
# 每次選手試跳後，呼叫 `record_jump(athlete_name, height)` 進行評比。
# 請補齊程式碼中的 `___`：
# 1. 宣告 global 變數 highest_jump 與 champion。
# 2. 使用 max() 或條件分支更新全域最高紀錄。
# ==========================================

highest_jump = 0.0
champion = ""

def record_jump(athlete_name, height):
    # 提示：宣告要修改的全域變數
    global highest_jump, ___
    
    if height > highest_jump:
        # 提示：刷新最高跳高紀錄與冠軍姓名
        highest_jump = ___
        champion = athlete_name

# 主程式測試
record_jump("小明", 1.65)
record_jump("小華", 1.78)
record_jump("小美", 1.72)

print(f"冠軍選手: {champion}，最高成績: {highest_jump} 公尺")
# 預期輸出: 冠軍選手: 小華，最高成績: 1.78 公尺

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.2
# 任務說明：
# 請設計一個物流公司「最低報價評估器」：
# 1. 全域變數 `min_cost = float('inf')`。
# 2. 定義函數 `submit_quote(cost)`：
#    - 透過 global 修改 min_cost。
#    - 利用 `min(min_cost, cost)` 更新全域最低成本。
# 3. 主程式依據測試資料連續送入報價，並印出最終最低運費。
#
# 【公開測試資料 1】
# 連續報價：submit_quote(150), submit_quote(120), submit_quote(180)
# 預期輸出：
# 全域最低運費: 120 元
#
# 【公開測試資料 2】
# 連續報價：submit_quote(90), submit_quote(110), submit_quote(85), submit_quote(95)
# 預期輸出：
# 全域最低運費: 85 元
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.2
# 任務說明：
# 請設計一個氣象監測站「全域極端氣溫監控系統」：
# 1. 同時在全域宣告 `max_temp = -float('inf')` 與 `min_temp = float('inf')`。
# 2. 定義函數 `record_temperature(temp)`：
#    - 使用 global 同時更新最高溫與最低溫。
# 3. 定義輔助函數 `get_temperature_range()`：
#    - 回傳全域全距溫差（max_temp - min_temp）。
# 4. 在主程式中連續輸入 5 筆氣溫，並輸出全域最高溫、最低溫與極端溫差。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.5.3 APCS 競賽實戰場景二：大型地圖 / 圖論鄰接矩陣的全域共享存取

#### 1. 生活故事比喻：指揮中心牆上的巨幅作戰地圖
想像一支特種作戰部隊派遣了 10 個偵察小組深入迷宮森林（呼叫多個函數或遞迴探測）。森林的地形極為複雜，畫在一張長寬各兩公尺的巨幅軍用地圖上（$1000 \times 1000$ 的二維網格）。
如果指揮官要求「每個小組出發時，都必須在背包裡塞一張等比例複印的巨幅地圖」（在函數參數中反覆複製或傳遞大型陣列），士兵們的背包立刻會被厚重的地圖塞爆，走不到兩步就體力衰竭當場倒地（這就是記憶體超限 Memory Limit Exceeded 與時間超限 Time Limit Exceeded）！
最明智的做法是什麼？指揮部把巨幅地圖高高懸掛在全域戰情室的正中央牆面上。所有前線小組透過無線電回報座標 `(r, c)`，指揮部抬頭看一眼牆上的地圖，立刻就知道那個座標是泥沼、河流還是空地。

#### 2. 底層運作機制：全域二維陣列的唯讀存取與 LEGB 原則
在 Python 中，全域宣告的二維串列 `grid = [[...], ...]` 位於全域作用域。
根據 Python 的 **LEGB 查找規則（Local $\rightarrow$ Enclosing $\rightarrow$ Global $\rightarrow$ Built-in）**：
- 當函數內部**僅僅是讀取** `grid[r][c]` 的元素數值時，區域找不到 `grid`，Python 就會自然向外查找全域命名空間，並順利讀取到全域地圖！
- **關鍵原理**：若函數內部只讀不改寫整個 `grid` 變數名稱，**完全不需要宣告 `global grid`**！
- 這樣做有兩大好處：
  1. **零拷貝高效率**：完全不會產生重複複製陣列的記憶體與時間開銷。
  2. **參數列清爽**：走訪函數的參數只需要傳遞當前座標 `(r, c)` 或步數 `step`，不需要每次都拖帶龐大的地圖參數。

#### 3. 初學者常見陷阱：邊界越界與型態混淆
在唯讀存取全域二維陣列時，最容易引發程式崩潰的不是變數名稱問題，而是**座標越界（`IndexError: list index out of range`）**！
在存取 `grid[r][c]` 之前，務必搭配合法的網格邊界防護條件：
```python
if 0 <= r < R and 0 <= c < C:
    val = grid[r][c]  # 安全讀取
```
若未檢查邊界，當探測走到網格邊緣時，程式會立刻在評判系統上遭遇 Runtime Error (RE)。

#### 4. APCS 實戰視野
APCS 歷屆實作題中，例如 e287（機器人的路徑）、g596（骨牌遊戲）、b266（矩陣轉換），地圖尺寸動輒包含成百上千個格子。將大型網格宣告在全域，並由子函數直接唯讀查詢，是競賽高分選手普遍採用的標準架構。

In [ ]:
# 範例 11.5.3：大型二維網格的全域共享存取與唯讀探測

# 在全域定義 4x4 的迷宮地圖（0: 平地, 1: 障礙物, 9: 寶藏）
grid = [
    [0, 0, 1, 0],
    [1, 0, 0, 0],
    [0, 1, 9, 1],
    [0, 0, 0, 0]
]
R = len(grid)
C = len(grid[0])

def inspect_cell(r, c):
    # 依據 LEGB 規則，此處僅讀取全域 grid，不需要宣告 global grid！
    # 嚴密邊界防護
    if not (0 <= r < R and 0 <= c < C):
        return "【越界非法座標】"
    
    cell_type = grid[r][c]
    if cell_type == 0:
        return "平地 (可通行)"
    elif cell_type == 1:
        return "障礙物 (無法通過)"
    elif cell_type == 9:
        return "🏆 發現稀世寶藏！"

# 模擬前線偵察探測各座標
print(f"[探測 (0, 1)] -> {inspect_cell(0, 1)}")
print(f"[探測 (2, 2)] -> {inspect_cell(2, 2)}")
print(f"[探測 (1, 0)] -> {inspect_cell(1, 0)}")
print(f"[探測 (4, 4)] -> {inspect_cell(4, 4)}")

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.3
# 任務說明：
# 某位同學正在設計一個全域網格障礙物計數器。
# 全域定義二維陣列 `board`（字元網格，'.' 為通道，'#' 為牆壁）。
# 函數 `count_adjacent_walls(r, c)` 負責統計 `(r, c)` 上下左右相鄰四格中牆壁 '#' 的數量。
# 請補齊程式碼中的 `___`：
# 1. 讀取全域網格行數與列數。
# 2. 走訪四方向位移，並在合法邊界內讀取全域 board 進行累計。
# ==========================================

board = [
    ['.', '#', '.'],
    ['#', '.', '#'],
    ['.', '#', '.']
]
H = len(board)
W = len(board[0])

def count_adjacent_walls(r, c):
    # 上、下、左、右四方向位移向量
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    walls = 0
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        # 提示：嚴密邊界檢查，合法範圍內才讀取全域 board
        if 0 <= nr < H and 0 <= nc < ___:
            if board[nr][nc] == ___:
                walls += 1
    return walls

# 主程式測試中心點 (1, 1) 的周圍牆壁數
center_walls = count_adjacent_walls(1, 1)
print(f"中心點周圍相鄰牆壁總數: {center_walls}")  # 預期輸出: 4

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.3
# 任務說明：
# 請設計一個「全域路徑總分計算器」：
# 1. 全域二維矩陣 `score_grid` 如下：
#    score_grid = [
#        [3, 1, 4],
#        [1, 5, 9],
#        [2, 6, 5]
#    ]
# 2. 定義函數 `calculate_path_score(coords)`：
#    - 接收一個座標列表 coords，例如 `[(0, 0), (1, 1), (2, 2)]`。
#    - 依序讀取全域 score_grid 中對應格子的數值並加總。
#    - 回傳累計總分。
#
# 【公開測試資料 1】
# 傳入路徑：[(0, 0), (1, 1), (2, 2)]
# 預期輸出：
# 路徑得分: 13  (3 + 5 + 5)
#
# 【公開測試資料 2】
# 傳入路徑：[(0, 1), (0, 2), (1, 2)]
# 預期輸出：
# 路徑得分: 14  (1 + 4 + 9)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.3
# 任務說明：
# 請設計一個全域網格高低起伏探測系統：
# 1. 全域定義 4x4 的地形高度矩陣 `elevation_map`（數值代表海拔高度）。
# 2. 定義函數 `find_local_peaks()`：
#    - 走訪全域地形矩陣的每一個格子（不含最外圍邊界）。
#    - 若某格子的海拔高度嚴格大於其上下左右 4 個相鄰格子的海拔，則該格為「局部山峰（Peak）」。
#    - 收集所有山峰座標 `(r, c)` 並以列表回傳。
# 3. 主程式印出偵測到的所有山峰位置。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.5.4 可變物件免 `global` 原理：為何串列原處修改不需要宣告 `global`？

#### 1. 生活故事比喻：黑板上的代辦清單 vs 換掉整面黑板
你在學校教室的黑板上畫了一張「每日值日生工作表」（全域串列 `tasks = ["擦窗戶", "掃地"]`）。
此時班長走進教室，想在表格最下方加上一項「倒垃圾」（呼叫 `tasks.append("倒垃圾")`），或者把第一項改成「擦黑板」（執行 `tasks[0] = "擦黑板"`）。
請問班長需要打電話給總務處申請「更換全教室黑板工程」（宣告 `global`）嗎？完全不需要！因為黑板依然是原本那塊黑板，班長只是在既有的黑板表面上塗塗改改（原處修改 In-place Modification）。
但是，如果校長走進教室說：「把這面舊黑板徹底拆除丟掉，換上一面全新的液晶電子白板！」（執行 `tasks = []` 重新賦值），這才叫做「換掉黑板本體」，必須有學校最高層的特許公文（宣告 `global`）！

#### 2. 底層運作機制：原處修改（In-place）vs 重新綁定（Rebinding）
這是 Python 物件模型中最核心也最優雅的機制之一：
- **重新賦值（Rebinding）**：
  若在函數內寫 `arr = []` 或 `arr = [1, 2, 3]`，這是使用等號 `=` 將變數名稱指向一個全新的物件。Python 直譯器在編譯期會認定 `arr` 是函數專屬的區域變數。如果沒加 `global arr`，外部全域變數絲毫不受影響！
- **原處修改（In-place Modification）**：
  若在函數內執行 `arr.append(x)`、`arr.pop()`、`arr.clear()` 或 `arr[i] = val`，變數名稱 `arr` 所指向的記憶體位址（Object Reference）**完全沒有改變**！你只是透過該位址呼叫了該串列內部的方法。
  根據 LEGB 原則，Python 找到全域的 `arr` 後，直接修改了該物件內部的內容。因此，**可變物件（串列、字典、集合）的原處修改，完全不需要宣告 `global`**！

#### 3. 初學者常見陷阱：想要清空全域串列卻寫錯語法
許多初學者在寫競賽程式時，想在函數內清空全域串列，卻順手寫了：
```python
def clear_data():
    data = []  # 致命錯誤！這只是建立了一個區域空串列，全域的 data 依然滿滿是舊資料！
```
如果想要在不使用 `global` 的前提下正確清空全域串列，有兩種標準寫法：
1. **使用方法**：`data.clear()`（極度推薦，直觀且高效）。
2. **使用切片清空**：`data[:] = []`（同樣直接清空原處記憶體）。

#### 4. APCS 實戰視野
理解「可變物件免 global 原處修改」，能讓你在實作拜訪標記陣列（`visited`）、路徑暫存隊列（`path`）或頻率字典時，保持代碼簡潔明快，避免畫蛇添足地寫下一堆不必要的 `global` 宣告。

In [ ]:
# 範例 11.5.4：可變物件原處修改 vs 重新綁定的深度實驗

# 全域串列
record_list = [10, 20]

def demo_in_place():
    # 原處修改：不需要 global！
    record_list.append(30)
    record_list[0] = 99
    print(f"  [原處修改內部] record_list id: {id(record_list)}")

def demo_wrong_rebind():
    # 錯誤示範：沒有宣告 global 卻進行賦值
    # Python 會建立一個名為 record_list 的區域變數，遮蔽外部全域變數！
    record_list = [1, 2, 3]
    print(f"  [區域重新賦值] 區域 record_list: {record_list} (id: {id(record_list)})")

print(f"[主程式初始] record_list: {record_list} (id: {id(record_list)})")

demo_in_place()
print(f"[呼叫 demo_in_place 後] 全域成功被修改: {record_list}\n")

demo_wrong_rebind()
print(f"[呼叫 demo_wrong_rebind 後] 全域完全沒被換掉: {record_list}")

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.4
# 任務說明：
# 某位同學在設計一個全域列印排隊系統 `print_queue`。
# 他希望在函數內提供加入工作與清空佇列的功能。
# 請補齊程式碼中的 `___`：
# 1. 在 `add_job(job_name)` 中使用 append() 原處加入工作。
# 2. 在 `clear_all_jobs()` 中使用 clear() 原處清空全域佇列（免 global）。
# ==========================================

print_queue = ["文件A.pdf", "照片B.png"]

def add_job(job_name):
    # 提示：原處追加新工作，不需要宣告 global
    print_queue.___(job_name)

def clear_all_jobs():
    # 提示：呼叫清空方法，原處抹除所有排隊工作
    print_queue.___()

# 主程式測試
add_job("作業C.docx")
print(f"目前排隊清單: {print_queue}")  # 預期輸出: ['文件A.pdf', '照片B.png', '作業C.docx']

clear_all_jobs()
print(f"清空後排隊清單: {print_queue}")  # 預期輸出: []

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.4
# 任務說明：
# 請設計一個「全域購物車動態管理系統」：
# 1. 全域宣告一個空串列 `cart = []`。
# 2. 定義函數 `add_item(item)`：
#    - 原處將商品 item 加入 cart 尾端。
# 3. 定義函數 `remove_first_item()`：
#    - 若 cart 非空，原處彈出並回傳第 0 個商品；若為空則回傳 None。
# 4. 依照兩組公開測試資料驗證，不需要使用任何 global 關鍵字。
#
# 【公開測試資料 1】
# 呼叫順序：add_item("筆記本"), add_item("鋼筆"), remove_first_item()
# 預期輸出：
# 移除商品: 筆記本
# 購物車剩餘: ['鋼筆']
#
# 【公開測試資料 2】
# 呼叫順序：add_item("尺"), add_item("橡皮擦"), add_item("圓規")
# 預期輸出：
# 購物車剩餘: ['尺', '橡皮擦', '圓規']
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.4
# 任務說明：
# 請設計一個「全域字典之學生成績簿」：
# 1. 全域定義字典 `grade_book = {}`。
# 2. 定義函數 `update_grade(student, score)`：
#    - 在字典中原處新增或修改該學生的成績。
# 3. 定義函數 `calculate_class_average()`：
#    - 唯讀存取全域 grade_book，計算並回傳全班平均分數（保留一位小數）。
# 4. 在主程式新增 4 位學生成績，並印出字典內容與全班平均分數。
# 5. 在註解中簡要說明為什麼對全域字典進行鍵值修改同樣不需要宣告 global。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.5.5 濫用 `global` 的隱患：狀態污染、跨測資未清空（重測致命傷）

#### 1. 生活故事比喻：考卷回收箱裡殘留的前一場試卷
想像一間考試教室的講台上，放著一個「考卷回收箱」（全域串列 `collected_papers = []`）。
第一場考試結束，3 位考生交了試卷放進箱子，監考老師抱起箱子統計人數，確認正好 3 張，成績無誤。接著第二場考試開始了，監考老師卻**忘了把箱子裡的第一場試卷清空**！第二場考試又有 2 位考生交卷，此時箱子裡竟然累積了 5 張試卷！電腦閱卷系統一算：「咦？這場考試明明只有 2 個人考，怎麼跑出 5 張試卷？」當場判定試場作弊，第二場所有考生全部被判 0 分！
這就是競技程式中最慘烈的悲劇——**「跨測資全域變數污染（Cross-testcase State Contamination）」**。

#### 2. 底層運作機制：全域變數的永久生命週期
在 Python 執行程序（Process）存活期間，全域變數具有「跨函數呼叫、甚至跨迴圈回合」的永久生命週期：
- 當線上評判系統（Online Judge, 如 ZeroJudge、APCS 官方系統）評判你的程式時，系統通常會在**同一次程式執行中，連續輸入多組測試資料**。
- 如果你的程式碼架構如下：
  ```python
  total_sum = 0  # 只在程式最頂端初始化了一次！
  
  def solve():
      global total_sum
      # 讀取當前測資並累加進 total_sum
  ```
  當系統執行第 1 筆測資時，`total_sum` 算出了 100（答對）；
  當系統緊接著餵入第 2 筆測資時，`total_sum` **並沒有自動歸零**！它依然是 100，第 2 筆測資算出來的 50 加進去變成了 150（標準答案應為 50），系統立刻判你 Wrong Answer (WA)！

#### 3. 初學者的重大困惑：為什麼本地端全對，上傳卻 0 分？
初學者在自己的電腦上練習時，往往是「按一次執行按鈕，鍵入第 1 組測資，看結果正確；再按一次執行按鈕，鍵入第 2 組測資，看結果又正確」。
初學者以為程式完全正確，但其實每一次手動重新執行，作業系統都會為 Python 開啟全新進程，變數自然被重新初始化。
然而，競賽評判伺服器是「一次執行、連續讀入全部測資」。只要有任何一個全域計數器或陣列沒有手動清空，從第 2 筆測資開始就會全面崩盤！

#### 4. APCS 實戰視野
診斷「跨測資污染」是成為成熟競賽選手的成年禮。每當你在 APCS 線上系統遇到「子任務 1 通過，但多筆測資子任務全數 WA」的現象時，第一個要懷疑的嫌疑犯就是全域變數未清空！

In [ ]:
# 範例 11.5.5：模擬跨測資污染的慘劇與對照實驗

# 【反面教材：未清空全域變數的污染版】
polluted_total = 0

def solve_polluted(numbers):
    global polluted_total
    for num in numbers:
        polluted_total += num
    return polluted_total

print("=== 模擬未清空全域變數的污染執行 ===")
test_case_1 = [10, 20, 30]
ans1 = solve_polluted(test_case_1)
print(f"第 1 筆測資結果: {ans1} (正確答案應為 60)")

test_case_2 = [5, 5]
ans2 = solve_polluted(test_case_2)
print(f"第 2 筆測資結果: {ans2} ❌ 慘遭污染！(正確答案應為 10，卻輸出了 70！)\n")


# 【正確防禦：每次運算前重置全域變數】
safe_total = 0

def solve_safe(numbers):
    global safe_total
    safe_total = 0  # ⭐ 關鍵防線：運算前強制歸零重置！
    for num in numbers:
        safe_total += num
    return safe_total

print("=== 模擬落實重置防禦的安全執行 ===")
print(f"第 1 筆測資結果: {solve_safe(test_case_1)} (正確: 60)")
print(f"第 2 筆測資結果: {solve_safe(test_case_2)} (正確: 10)")

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.5
# 任務說明：
# 某位同學寫了一個收集訪客名單的函數 `log_visitors(names)`。
# 全域串列 `visitor_log` 記錄當天所有訪客姓名。
# 由於忘記在每次換日（新測資）時清空，導致名單跨日重複累積。
# 請補齊程式碼中的 `___`：
# 1. 在換日處理前，呼叫正確的方法將全域 visitor_log 清空。
# 2. 將新一批訪客加入清單並回傳清單長度。
# ==========================================

visitor_log = []

def process_daily_batch(names):
    # 提示：在處理每日新批次前，必須徹底清空舊紀錄！
    visitor_log.___()
    
    for name in names:
        visitor_log.append(name)
    return len(visitor_log)

# 主程式測試連續兩天
day1_guests = ["Alice", "Bob"]
day2_guests = ["Charlie"]

count1 = process_daily_batch(day1_guests)
print(f"第 1 天訪客數: {count1}")  # 預期: 2

count2 = process_daily_batch(day2_guests)
print(f"第 2 天訪客數: {count2}")  # 預期: 1（若污染會變成 3）

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.5
# 任務說明：
# 請設計一個字元出現次數統計系統，並防禦跨測資污染：
# 1. 全域變數 `char_count = 0`。
# 2. 定義函數 `count_letter(target_str, search_char)`：
#    - 每次執行統計前，必須將全域 char_count 重置歸零！
#    - 統計 target_str 中 search_char 出現的次數並存入 char_count。
#    - 回傳 char_count。
# 3. 模擬連續處理兩筆字串測資，驗證第 2 筆測資是否維持獨立純淨。
#
# 【公開測試資料 1】
# 呼叫：count_letter("banana", "a")
# 預期輸出：
# 字母出現次數: 3
#
# 【公開測試資料 2】
# 接續呼叫：count_letter("apple", "p")
# 預期輸出：
# 字母出現次數: 2  (若污染會變成 5)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.5
# 任務說明：
# 請設計一個模擬 Online Judge 自動評判批改系統：
# 1. 題目要求計算數列中「偶數的總和」。
# 2. 全域變數 `even_sum = 0`。
# 3. 準備 3 組數列測資：
#    - 測資 1: [2, 4, 6] （答案 12）
#    - 測資 2: [1, 3, 5] （答案 0）
#    - 測資 3: [10, 20]  （答案 30）
# 4. 請分別實作「未清空污染版函數」與「安全防禦版函數」，在同一個主程式中依序批改這 3 組測資，對比印出兩者的批改結果。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

### 11.5.6 競賽除錯守則：如何在多筆測資迴圈開頭正確重置（Reset）全域變數

#### 1. 生活故事比喻：手術室的無菌重置標準作業程序（SOP）
在一間專業的外科手術室中，每當一位病患手術成功推出手術室、準備迎接下一位病患進場時，醫護團隊必須執行最嚴格的「無菌重置 SOP」：所有使用過的手術刀丟棄或高溫消毒，手術台上的無菌鋪單全部換新，器械盤徹底清洗歸零。絕不可能帶著上一位病患的血跡，直接替下一位病患開刀！
在競賽程式設計的世界裡，每一筆測試資料就是一位全新的病患。你的程式碼必須在讀入每筆測資的「第一秒鐘」，毫不留情地執行全面性的「重置（Reset）SOP」。

#### 2. 底層運作機制：多筆測資的黃金重置架構
處理 APCS 多筆測資題目時，有兩種最標準且防禦力極強的架構：

##### 架構模式 A：專屬 `reset()` 函數模式（最清晰）
```python
def reset():
    global ans, step_count
    ans = float('inf')
    step_count = 0
    visited_grid.clear()

def solve():
    reset()  # ⭐ 每一筆測資開始處理的第一行，無條件執行重置！
    # 讀取輸入、運算並輸出
```

##### 架構模式 B：測資迴圈頂端就地重置（極速實用）
```python
T = int(input())
for case_num in range(T):
    # ⭐ 迴圈內部最頂端：立刻宣告並初始化所有全域狀態
    ans = 0
    history.clear()
    
    # 執行該筆測資的解題邏輯
```

#### 3. 初學者常見陷阱：重置寫在「迴圈結尾」的巨大漏洞
有些初學者習慣把重置程式碼寫在迴圈的最後一行（`for` 迴圈結尾）。
這看似有道理，但隱藏著極度危險的致命陷阱：
如果在解題過程中，遇到了特殊情況需要提早跳過（例如 `if invalid_input: continue`），或者發生異常提前退出，**寫在結尾的重置程式碼就會被直接跳過**！下一回合的測資就會無預警吞下上一回合的污染資料！
因此請牢記競賽除錯鐵律：**「重置代碼永遠只能放在迴圈開頭（Loop Entrance），絕不放結尾！」**

#### 4. APCS 實戰視野
掌握這個競賽除錯守則，能讓你兼具全域變數「快速直覺、免傳多參數」的編碼優勢，同時擁有區域變數「純淨無暇、互不干擾」的安全性。在面對包含 20 筆隱藏測試資料的 APCS 大題時，保證每一筆測資都能獨立乾淨地運行，穩定奪得滿分 Accepted (AC)！

In [ ]:
# 範例 11.5.6：APCS 多筆測資黃金重置模板演示

# 全域狀態變數
max_score = 0
passed_students = []

def reset_state():
    """重置全域狀態 SOP，保證每筆測資乾淨無污染"""
    global max_score
    max_score = 0
    passed_students.clear()

def process_testcase(case_id, score_list):
    # 1. 第一步：無條件執行狀態重置
    reset_state()
    
    # 2. 第二步：執行當前測資的業務邏輯
    global max_score
    for name, score in score_list:
        if score > max_score:
            max_score = score
        if score >= 60:
            passed_students.append(name)
            
    # 3. 輸出該筆測資結果
    print(f"--- 測資組 #{case_id} 結算 ---")
    print(f"  最高分: {max_score}")
    print(f"  及格名單: {passed_students}")

# 模擬 APCS 連續灌入 2 筆測資
batch_1 = [("小明", 55), ("小華", 88), ("小美", 70)]
process_testcase(1, batch_1)

batch_2 = [("阿強", 40), ("大雄", 59)]
process_testcase(2, batch_2)

In [ ]:
# ==========================================
# [3] Code 填空題 11.5.6
# 任務說明：
# 某位同學正在模擬競賽多筆測資輸入。
# 題目要求統計每組資料中「奇數的總和」與「奇數個數」。
# 請補齊程式碼中的 `___`，落實開頭重置 SOP：
# 1. 在 `reset_contest()` 中將全域 odd_sum 與 odd_count 歸零。
# 2. 在 `solve_case(numbers)` 最開頭呼叫重置函數。
# ==========================================

odd_sum = 0
odd_count = 0

def reset_contest():
    global odd_sum, odd_count
    # 提示：將全域計數器與累加器歸零
    odd_sum = ___
    odd_count = ___

def solve_case(numbers):
    # 提示：處理測資前第一步，呼叫重置函數！
    ___()
    
    global odd_sum, odd_count
    for x in numbers:
        if x % 2 != 0:
            odd_sum += x
            odd_count += 1
    return odd_sum, odd_count

# 主程式測試兩組測資
ans1 = solve_case([1, 2, 3, 4, 5])
print(f"第 1 組奇數和與數量: {ans1}")  # 預期: (9, 3)

ans2 = solve_case([2, 4, 6])
print(f"第 2 組奇數和與數量: {ans2}")  # 預期: (0, 0)

In [ ]:
# ==========================================
# [4] Code 練習題 11.5.6
# 任務說明：
# 請撰寫一個多筆測資的「最大溫差分析器」：
# 1. 定義全域變數 `daily_high = -float('inf')` 與 `daily_low = float('inf')`。
# 2. 定義 `reset_day()` 函數重置上述兩個全域變數。
# 3. 定義 `analyze_day(temperatures)` 函數：
#    - 開頭呼叫 reset_day()。
#    - 找出當天的最高溫與最低溫，回傳最大溫差（daily_high - daily_low）。
# 4. 主程式連續分析兩天資料並輸出。
#
# 【公開測試資料 1】
# 第 1 天氣溫：[20, 28, 18, 31, 25]
# 預期輸出：
# 第 1 天最大溫差: 13 度
#
# 【公開測試資料 2】
# 第 2 天氣溫：[15, 17, 16, 15]
# 預期輸出：
# 第 2 天最大溫差: 2 度
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.5.6
# 任務說明：
# 請設計一個完整的多筆測資競技程式架構：
# 1. 模擬線上題庫輸入格式：
#    - 輸入多筆測資，每筆測資包含多名參賽者的 (姓名, 得分)。
# 2. 全域狀態包括：`champion_name`、`champion_score`、以及全體參賽者分數串列 `all_scores`。
# 3. 實作規範：
#    - 封裝 `init_testcase()` 負責開頭全面清空重置。
#    - 封裝 `record_score(name, score)` 記錄選手。
#    - 封裝 `finalize_report()` 計算並回傳該場比賽冠軍與全體平均分。
# 4. 模擬連續執行 3 場不同人數的比賽，驗證各場比賽狀態徹底獨立不污染。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：

## 11.5 單元重點回顧與自我檢驗

恭喜你完成了第 11.5 單元的全面修練！在程式設計的世界中，全域變數就像一把無比鋒利的雙刃劍：用得好，能讓複雜的演算法代碼精簡一半、執行如飛；用得不慎，則可能引發難以排查的狀態污染，在競賽中痛失分數。掌握了本單元的防禦守則，你已經具備了專業競技選手的敏銳嗅覺。

### 核心觀念複習清單
1. **`global` 關鍵字機制**：
   - 在函數內部宣告 `global x`，打破區域作用域城牆，賦予內部重新賦值改寫全域變數的權限。
   - 語法鐵律：必須單獨宣告，不可寫成單行賦值 `global x = 10`。
2. **APCS 最佳解累計模式**：
   - 透過全域維護 `ans = min(ans, ...)` 或 `max()`，讓深層搜尋函數直達解題目標，免去繁複的多層回傳。
   - 牢記極值初始化原則：找最小設無限大 `float('inf')`，找最大設極小值。
3. **大型地圖共享與唯讀存取**：
   - 依據 LEGB 原則，函數若僅僅讀取全域大型陣列 `grid[r][c]`，**完全不需要宣告 `global`**。
   - 節省記憶體並杜絕陣列傳遞的效能浩劫，搭配嚴密邊界防護遠離 `IndexError`。
4. **可變物件免 `global` 原處修改**：
   - `arr.append()`、`arr.pop()`、`arr[i] = val`、`arr.clear()` 是對記憶體中既有物件內部操作，不需要 `global`。
   - 等號賦值 `arr = []` 是重新綁定，沒加 `global` 只是建立了區域變數。
5. **跨測資污染的診斷與警惕**：
   - 全域變數具有永久生命週期。若多筆測資未清空，舊資料殘留會引發連環 Wrong Answer (WA)。
6. **競賽重置黃金 SOP**：
   - 重置代碼永遠只能放在「每筆測資迴圈開頭（Loop Entrance）」，絕不可放結尾，徹底杜絕中途跳出造成的防禦漏洞。

---

### 下一步精彩預告
我們已經掌握了函數的定義、參數傳遞、變數範圍，以及全域變數的競賽實戰應用。接下來，我們要邁入電腦科學中最神奇、最具藝術感，同時也是 APCS 觀念題與實作題最核心的思維分水嶺——**遞迴（Recursion）**！
- 函數竟然可以在自己的肚子裡「呼叫自己」？這不會陷入無窮死結嗎？
- 電腦底層的「呼叫堆疊（Call Stack）」是如何一層層壓入又一層層彈出的？
- 遞迴的煞車皮「終止條件（Base Case）」該如何設計？

請緊接著邁向 **[11.6 線性遞迴（Linear Recursion）：呼叫堆疊（Call Stack）展開與終止條件]**，開啟你的遞迴思維大冒險！